Обучение на сохранённом `data/letters.pt`. Датасет создаётся отдельно в `letters_dataset.ipynb`.


In [ ]:
from pathlib import Path
import os
import csv
import math
import sys
import time

import numpy as np
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import torch
from torch.nn import functional as F

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data" / "letters.pt"
RESULTS = ROOT / "results" / "letters_scale_epochwise"
SEED = 42
TARGET = "H"
TRAIN_PER_CLASS = 128
BATCH_SIZE = 64
MAX_EPOCHS = 120
CHECK_EVERY = 5
TARGET_ACCURACY = 0.99
LR = 5e-4
WEIGHT_DECAY = 0.01
DONOR_COUNTS = (20, 25)

assert torch.cuda.is_available(), "Нужна CUDA-сборка PyTorch."
DEVICE = torch.device("cuda")
torch.set_num_threads(2)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark = False
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from model import DeiTTiny
print(torch.cuda.get_device_name(0))


In [ ]:
data = torch.load(DATA_PATH, map_location="cpu", weights_only=True)
images = data["images"]
LETTERS = data["letters"]
HEIGHTS = data["heights"]
CANVAS = data["canvas"]
pools = data["shift_pools"]
TEST_POSITIONS = len(pools[0]["test"])
assert all(len(pool["test"]) == TEST_POSITIONS for pool in pools)
print("Loaded:", DATA_PATH, tuple(images.shape))


При обучении сдвиги выбираются независимо для каждого изображения из train-координат.
Генераторы создаются один раз на запуск: новые батчи получают новые сдвиги; случайные совпадения возможны.
Между экспериментами начальные веса, порядок примеров и последовательности случайных чисел повторяются.
Число больших классов задаётся в DONOR_COUNTS. H никогда не бывает большой в обучении.
У выбранных классов половина примеров в каждой эпохе большая. Всего 3328 примеров на эпоху.

In [14]:
templates = images.to(DEVICE, dtype=torch.float32).div(255)


def make_batch(labels, sizes, shifts):
    labels, sizes, shifts = labels.to(DEVICE), sizes.to(DEVICE), shifts.to(DEVICE)
    source = templates[labels, sizes]
    theta = torch.eye(2, 3, device=DEVICE).unsqueeze(0).repeat(len(labels), 1, 1)
    theta[:, 0, 2] = -2 * shifts[:, 0] / CANVAS
    theta[:, 1, 2] = -2 * shifts[:, 1] / CANVAS
    grid = F.affine_grid(theta, source.shape, align_corners=False)
    shifted = F.grid_sample(source, grid, mode="nearest", padding_mode="zeros", align_corners=False)
    return shifted.expand(-1, 3, -1, -1)


def classification_metrics(logits, labels):
    log_probabilities = logits.log_softmax(-1)
    probabilities = log_probabilities.exp()
    predicted = logits.argmax(-1)
    correct = predicted.eq(labels)
    true_probability = probabilities.gather(1, labels[:, None]).squeeze(1)
    return {
        "loss": F.cross_entropy(logits, labels).item(),
        "accuracy": correct.float().mean().item(),
        "true_confidence": true_probability.mean().item(),
        "entropy": -(probabilities * log_probabilities).sum(-1).mean().item(),
        "confidence_correct": true_probability[correct].mean().item() if correct.any() else float("nan"),
        "correct": correct.sum().item(), "count": len(labels),
        "predicted_counts": torch.bincount(predicted, minlength=len(LETTERS)).tolist(),
        "probabilities": probabilities,
    }


@torch.inference_mode()
def evaluate(model, classes, size_index, split):
    model.eval()
    coordinates = pools[size_index][split]
    labels = torch.tensor(classes).repeat_interleave(len(coordinates))
    shifts = coordinates.repeat(len(classes), 1)
    logits = []
    for start in range(0, len(labels), BATCH_SIZE):
        batch_labels = labels[start:start + BATCH_SIZE]
        sizes = torch.full_like(batch_labels, size_index)
        x = make_batch(batch_labels, sizes, shifts[start:start + BATCH_SIZE])
        logits.append(model(x).cpu())
    return classification_metrics(torch.cat(logits), labels)


def append_csv(path, row):
    exists = path.exists()
    with path.open("a", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=list(row))
        if not exists:
            writer.writeheader()
        writer.writerow(row)


torch.manual_seed(SEED)
initial_model = DeiTTiny(num_classes=len(LETTERS))
initial_weights = {key: value.detach().clone() for key, value in initial_model.state_dict().items()}
del initial_model
base_labels = torch.arange(len(LETTERS)).repeat_interleave(TRAIN_PER_CLASS)
donor_order = LETTERS.replace(TARGET, "")
all_classes = list(range(len(LETTERS)))
test_labels = np.concatenate([np.repeat(np.arange(len(LETTERS)), TEST_POSITIONS),
                              np.full(TEST_POSITIONS, LETTERS.index(TARGET))])
test_heights = np.concatenate([np.full(len(LETTERS) * TEST_POSITIONS, HEIGHTS[0]),
                               np.full(TEST_POSITIONS, HEIGHTS[1])])
test_shifts = np.concatenate([pools[0]["test"].numpy()] * len(LETTERS)
                            + [pools[1]["test"].numpy()])


AdamW, cross-entropy, до 120 эпох. Learning rate: 3 эпохи разогрева, затем cosine decay.
Каждую эпоху: train loss/accuracy; validation; фиксированный тест (все маленькие буквы и большая H).
Для теста сохраняем loss, accuracy/recall, вероятность истинного класса и энтропию (наты).
В *_probabilities.npz сохраняются вероятности [эпоха, изображение, предсказанный класс], метки, размеры,
сдвиги, порядок классов, доноры и число обучающих примеров каждого класса/размера за эпоху.
Один файл на запуск, обновляется каждую эпоху. float32, около 0.72 МБ на эпоху; изображения не дублируются.
Тестовые метрики служат диагностике и не влияют на остановку или обновление весов.
Остановка и сохранение весов раз в 5 эпох: два замера validation >=99% на маленьких и больших донорах.
Большая H не участвует в обучении и выборе момента остановки.
Новые результаты: results/letters_scale_epochwise; старые results/letters_scale сохраняются.
Повторное выполнение следующей ячейки перезапишет CSV и веса только в новой папке.


In [15]:
RESULTS.mkdir(parents=True, exist_ok=True)
metrics_path = RESULTS / "metrics.csv"
history_path = RESULTS / "history.csv"
for path in (metrics_path, history_path):
    if path.exists():
        path.unlink()
results = []

In [17]:


for donor_count in DONOR_COUNTS:
    run_name = f"big_{donor_count:02d}"
    donors = donor_order[:donor_count]
    donor_classes = [LETTERS.index(letter) for letter in donors]
    augmented = torch.tensor([letter in donors for letter in LETTERS])
    torch.manual_seed(SEED)
    model = DeiTTiny(num_classes=len(LETTERS)).to(DEVICE)
    model.load_state_dict(initial_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, fused=True)
    order_rng = torch.Generator().manual_seed(44)
    shift_rng = torch.Generator().manual_seed(45)
    size_rng = torch.Generator().manual_seed(46)
    consecutive = 0
    probability_history = []
    started = time.perf_counter()
    print(f"{run_name}, donors={donors or '-'}: start", flush=True)

    for epoch in range(1, MAX_EPOCHS + 1):
        factor = epoch / 3 if epoch <= 3 else 0.02 + 0.98 * (1 + math.cos(math.pi * (epoch - 3) / (MAX_EPOCHS - 3))) / 2
        for group in optimizer.param_groups:
            group["lr"] = LR * factor
        order = torch.randperm(len(base_labels), generator=order_rng)
        size_flags = torch.zeros(len(LETTERS), TRAIN_PER_CLASS, dtype=torch.bool)
        for c in all_classes:
            size_flags[c, torch.randperm(TRAIN_PER_CLASS, generator=size_rng)[:TRAIN_PER_CLASS // 2]] = True
        size_flags = size_flags.flatten()
        model.train()
        total_loss, total_correct = 0.0, 0
        for ids in order.split(BATCH_SIZE):
            labels = base_labels[ids]
            sizes = (augmented[labels] & size_flags[ids]).long()
            choices = [pool["train"][torch.randint(len(pool["train"]), (len(ids),), generator=shift_rng)] for pool in pools]
            shifts = torch.where(sizes[:, None].bool(), choices[1], choices[0])
            x = make_batch(labels, sizes, shifts)
            labels = labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = F.cross_entropy(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item() * len(ids)
            total_correct += logits.argmax(-1).eq(labels).sum().item()

        small_val = evaluate(model, all_classes, 0, "val")
        large_val = evaluate(model, donor_classes, 1, "val") if donor_classes else None
        small = evaluate(model, all_classes, 0, "test")
        large = evaluate(model, [LETTERS.index(TARGET)], 1, "test")
        probability_history.append(torch.cat([small["probabilities"], large["probabilities"]]).numpy())
        train_counts = np.column_stack([
            np.where(augmented.numpy(), TRAIN_PER_CLASS // 2, TRAIN_PER_CLASS),
            np.where(augmented.numpy(), TRAIN_PER_CLASS // 2, 0),
        ])
        np.savez(RESULTS / f"{run_name}_probabilities.npz",
                 probabilities=np.stack(probability_history), epochs=np.arange(1, epoch + 1),
                 labels=test_labels, heights=test_heights, shifts=test_shifts,
                 classes=np.array(list(LETTERS)), donors=np.array(list(donors), dtype="U1"),
                 train_counts=train_counts, train_heights=np.array(HEIGHTS), seed=SEED)
        large_seen_accuracy = large_val["accuracy"] if large_val else 1.0
        append_csv(history_path, {
            "run": run_name, "epoch": epoch, "lr": optimizer.param_groups[0]["lr"],
            "train_loss": total_loss / len(base_labels), "train_accuracy": total_correct / len(base_labels),
            "val_small_loss": small_val["loss"], "val_small_accuracy": small_val["accuracy"],
            "val_small_true_confidence": small_val["true_confidence"], "val_small_entropy": small_val["entropy"],
            "test_small_loss": small["loss"], "test_small_accuracy": small["accuracy"],
            "test_small_true_confidence": small["true_confidence"], "test_small_entropy": small["entropy"],
            "test_h_loss": large["loss"], "test_h_recall": large["accuracy"],
            "test_h_true_confidence": large["true_confidence"], "test_h_entropy": large["entropy"],
            "val_donor_large_accuracy": large_val["accuracy"] if large_val else float("nan"),
            "val_donor_large_loss": large_val["loss"] if large_val else float("nan"),
            "val_donor_large_true_confidence": large_val["true_confidence"] if large_val else float("nan"),
            "val_donor_large_entropy": large_val["entropy"] if large_val else float("nan"),
        })
        if epoch % CHECK_EVERY == 0 or epoch == MAX_EPOCHS:
            reached = small_val["accuracy"] >= TARGET_ACCURACY and large_seen_accuracy >= TARGET_ACCURACY
            consecutive = consecutive + 1 if reached else 0
            print(f"  epoch {epoch:03d}: train={total_correct / len(base_labels):.4f}, val small={small_val['accuracy']:.4f}, val donor large={large_seen_accuracy:.4f}, H p_true={large['true_confidence']:.5f}, elapsed={time.perf_counter() - started:.0f}s", flush=True)
            torch.save({"state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
                        "letters": LETTERS, "donors": donors, "seed": SEED, "epoch": epoch}, RESULTS / f"{run_name}.pt")
            if consecutive >= 2:
                break

    row = {
        "run": run_name, "donors": donors, "donor_count": donor_count, "target": TARGET,
        "seed": SEED, "device": str(DEVICE), "epochs": epoch, "images_seen": epoch * len(base_labels),
        "train_per_class": TRAIN_PER_CLASS, "batch_size": BATCH_SIZE, "learning_rate": LR,
        "val_small_accuracy": small_val["accuracy"], "val_donor_large_accuracy": large_val["accuracy"] if large_val else float("nan"),
        "val_target_met": consecutive >= 2, "small_accuracy": small["accuracy"], "large_recall": large["accuracy"],
        "small_true_confidence": small["true_confidence"], "large_true_confidence": large["true_confidence"],
        "small_entropy": small["entropy"], "large_entropy": large["entropy"],
        "small_confidence_correct": small["confidence_correct"], "large_confidence_correct": large["confidence_correct"],
        "small_loss": small["loss"], "large_loss": large["loss"],
        "small_correct": small["correct"], "small_count": small["count"], "large_correct": large["correct"], "large_count": large["count"],
        "large_predicted_counts": str(dict(zip(LETTERS, large["predicted_counts"]))),
    }
    append_csv(metrics_path, row)
    results.append(row)
    print(f"  TEST small={small['accuracy']:.4f}, large H recall={large['accuracy']:.4f}", flush=True)
    del model, optimizer
    if donor_count == 0 and not row["val_target_met"]:
        print("Baseline did not reach the validation target; adjust training before running donor experiments.", flush=True)
        break

big_20, donors=ABCDEFGIJKLMNOPQRSTU: start
  epoch 005: train=0.1229, val small=0.0385, val donor large=0.3016, H p_true=0.00754, elapsed=140s
  epoch 010: train=0.1313, val small=0.0391, val donor large=0.2672, H p_true=0.01904, elapsed=266s
  epoch 015: train=0.1989, val small=0.1821, val donor large=0.4242, H p_true=0.00770, elapsed=392s
  epoch 020: train=0.2945, val small=0.2188, val donor large=0.4410, H p_true=0.01110, elapsed=518s
  epoch 025: train=0.3933, val small=0.3630, val donor large=0.4109, H p_true=0.00996, elapsed=643s
  epoch 030: train=0.3534, val small=0.3978, val donor large=0.5035, H p_true=0.01151, elapsed=769s
  epoch 035: train=0.2816, val small=0.1436, val donor large=0.3449, H p_true=0.01450, elapsed=896s
  epoch 040: train=0.4180, val small=0.5141, val donor large=0.5586, H p_true=0.01064, elapsed=1022s
  epoch 045: train=0.4519, val small=0.5012, val donor large=0.5004, H p_true=0.00215, elapsed=1148s
  epoch 050: train=0.6635, val small=0.5604, val donor 

In [18]:
print(f"{'Run':10s} {'Epochs':>7s} {'Small acc':>10s} {'H recall':>10s} {'Small p_true':>13s} {'H p_true':>11s}")
for row in results:
    print(f"{row['run']:10s} {row['epochs']:7d} {row['small_accuracy']:10.4f} {row['large_recall']:10.4f} {row['small_true_confidence']:13.4f} {row['large_true_confidence']:11.6f}")
print("Saved:", RESULTS)


Run         Epochs  Small acc   H recall  Small p_true    H p_true
big_20          95     1.0000     0.0000        0.9969    0.003957
big_25         100     0.9997     0.0000        0.9965    0.000558
Saved: C:\Users\Mikhail\Desktop\projects\symmetry_priors\results\letters_scale_epochwise
